# Healthcare ML Tutorial: ICU Mortality Prediction

**Predicting In-Hospital Mortality for ICU Patients Using Structured Clinical Data from MIMIC-III**

This tutorial walks through a complete machine learning pipeline — from data extraction and preprocessing through traditional ML models (Logistic Regression, Random Forest, XGBoost) to a deep learning model (feedforward neural network) — with model comparison and clinical interpretability.

---

## Table of Contents

1. [Setup & Environment](#section-1-setup--environment)
2. [Data Loading & Exploration](#section-2-data-loading--exploration)
3. [Preprocessing & Cleaning](#section-3-preprocessing--cleaning)
4. [Feature Engineering](#section-4-feature-engineering)
5. [Traditional ML Models](#section-5-traditional-ml-models)
6. [Deep Learning Model](#section-6-deep-learning-model)
7. [Evaluation & Comparison](#section-7-evaluation--comparison)
8. [Interpretability / SHAP](#section-8-interpretability--shap)
9. [Bonus: Multimodal Extension (Optional)](#section-9-bonus-multimodal-extension-optional)
10. [Conclusion & References](#section-10-conclusion--references)

---

<a id="section-1-setup--environment"></a>

## 1. Setup & Environment

### Learning Objectives

By the end of this section, you will:

- Install all required Python packages for the tutorial
- Set random seeds across Python, NumPy, and PyTorch for **reproducible results**
- Verify your runtime environment (Python version, package versions, GPU availability)
- Understand why reproducibility and environment verification matter in healthcare ML research

### Prerequisites

This tutorial requires the following Python packages. The versions listed are the minimum tested versions; newer versions should also work.

| Package | Version | Purpose |
|---|---|---|
| `pandas` | ≥ 1.5.0 | Data manipulation and analysis |
| `numpy` | ≥ 1.23.0 | Numerical computing |
| `scikit-learn` | ≥ 1.2.0 | Traditional ML models and preprocessing |
| `xgboost` | ≥ 1.7.0 | Gradient boosted trees |
| `torch` | ≥ 2.0.0 | Deep learning (feedforward neural network) |
| `shap` | ≥ 0.42.0 | Model interpretability |
| `matplotlib` | ≥ 3.6.0 | Plotting and visualization |
| `seaborn` | ≥ 0.12.0 | Statistical data visualization |
| `transformers` | ≥ 4.30.0 | ClinicalBERT embeddings (bonus section) |
| `hypothesis` | ≥ 6.80.0 | Property-based testing |
| `google-cloud-bigquery` | ≥ 3.11.0 | Querying MIMIC-III data from Google BigQuery |

**Note:** This notebook is designed for [Google Colab](https://colab.research.google.com/). Some packages (NumPy, pandas, matplotlib, seaborn) come pre-installed in Colab, but we install them explicitly to ensure compatible versions.

In [ ]:
# ============================================================
# Install required packages
# ============================================================
# We use pip to install all dependencies. The --quiet flag
# reduces output noise so you can focus on any errors.
# google-cloud-bigquery is needed to query MIMIC-III data
# directly from Google BigQuery (physionet-data.mimiciii_clinical).
# ============================================================

!pip install --quiet \
    pandas \
    numpy \
    scikit-learn \
    xgboost \
    torch \
    shap \
    matplotlib \
    seaborn \
    transformers \
    hypothesis \
    google-cloud-bigquery

In [ ]:
# ============================================================
# Set random seeds for reproducibility
# ============================================================
# In healthcare ML research, reproducibility is critical.
# We set seeds for every source of randomness so that results
# are identical across runs. This covers:
#   - Python's built-in random module (used by some libraries)
#   - NumPy's random number generator (used by scikit-learn, pandas)
#   - PyTorch's CPU random generator (used by our neural network)
#   - PyTorch's CUDA random generator (if training on GPU)
# ============================================================

import random
import numpy as np
import torch

# Define a single seed constant used everywhere
SEED = 42

# Python built-in random
random.seed(SEED)

# NumPy random
np.random.seed(SEED)

# PyTorch CPU random
torch.manual_seed(SEED)

# PyTorch CUDA random (no-op if CUDA unavailable)
torch.cuda.manual_seed_all(SEED)

print(f"Random seeds set to {SEED} for Python, NumPy, and PyTorch (CPU + CUDA).")

In [ ]:
# ============================================================
# Environment verification
# ============================================================
# Verify that all packages are installed and check GPU availability.
# This catches setup issues early.
# ============================================================

import sys
import importlib

print(f"Python version: {sys.version}")
print("=" * 50)

packages = [
    ("pandas", "pandas"),
    ("numpy", "numpy"),
    ("sklearn", "scikit-learn"),
    ("xgboost", "xgboost"),
    ("torch", "torch"),
    ("shap", "shap"),
    ("matplotlib", "matplotlib"),
    ("seaborn", "seaborn"),
    ("transformers", "transformers"),
    ("hypothesis", "hypothesis"),
    ("google.cloud.bigquery", "google-cloud-bigquery"),
]

print("\nPackage Versions:")
print("-" * 50)
for import_name, display_name in packages:
    try:
        mod = importlib.import_module(import_name)
        version = getattr(mod, "__version__", getattr(mod, "VERSION", "installed"))
        print(f"  {display_name:.<30} {version}")
    except ImportError:
        print(f"  {display_name:.<30} NOT FOUND")

print("\n" + "=" * 50)
print("\nGPU Availability:")
print("-" * 50)
if torch.cuda.is_available():
    print(f"  CUDA is available")
    print(f"  GPU device: {torch.cuda.get_device_name(0)}")
    print(f"  CUDA version: {torch.version.cuda}")
else:
    print("  CUDA is NOT available -- training will use CPU")
    print("  Tip: In Colab, go to Runtime > Change runtime type > GPU")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"\nUsing device: {device}")
print("\nEnvironment setup complete!")

---

<a id="section-2-data-loading--exploration"></a>

## 2. Data Loading & Exploration

### Learning Objectives

By the end of this section, you will:

- Authenticate with Google Cloud and query MIMIC-III clinical data from BigQuery
- Understand the structure and clinical significance of each MIMIC-III source table
- Merge patient, admission, ICU stay, vital sign, and lab data into a single analysis-ready DataFrame
- Filter clinical events to the **first 24 hours** of each ICU stay (the standard prediction window)
- Compute basic summary statistics including patient counts, mortality rates, and data missingness
- Know where to find alternative datasets if you don't have MIMIC-III access

### MIMIC-III Source Tables

We use five core tables from the [MIMIC-III Clinical Database](https://physionet.org/content/mimiciii/1.4/) hosted on Google BigQuery:

| Table | Description | Clinical Significance |
|---|---|---|
| **PATIENTS** | Demographics (subject_id, gender, date of birth) | Provides baseline patient characteristics. Age and sex are known predictors of ICU mortality. |
| **ADMISSIONS** | Hospital admission details (admit/discharge times, admission type, insurance, mortality flag) | Contains the **target label** (`hospital_expire_flag`) and admission context that influences outcomes (e.g., emergency vs. elective). |
| **ICUSTAYS** | ICU stay records (ICU in/out times, length of stay) | Defines the **prediction window** -- we use the first 24 hours from `intime` to capture early physiological data. |
| **CHARTEVENTS** | Bedside vital sign measurements (heart rate, blood pressure, SpO2, temperature, respiratory rate) | High-frequency physiological signals recorded by nurses and monitors. Vital sign trends in the first 24h are strong mortality predictors. |
| **LABEVENTS** | Laboratory test results (WBC, hemoglobin, creatinine, lactate, etc.) | Objective biomarkers reflecting organ function. Abnormal labs (e.g., elevated lactate, low platelets) indicate severity of illness. |

Each table is linked by `subject_id` (patient) and `hadm_id` (hospital admission), allowing us to join them into a unified dataset.

### PhysioNet Data Use Agreement

> **Important:** MIMIC-III is a restricted-access dataset. Before using this data, you must:
>
> 1. Complete the [CITI "Data or Specimens Only Research" training](https://physionet.org/settings/credentialing/)
> 2. Sign the PhysioNet Credentialed Health Data Use Agreement
> 3. Request access to MIMIC-III on [PhysioNet](https://physionet.org/content/mimiciii/1.4/)
> 4. Link your PhysioNet account to your Google account for BigQuery access
>
> By using this data, you agree to:
> - Use the data only for legitimate research purposes
> - Not attempt to re-identify any individual patients
> - Not share the data with unauthorized users
> - Cite the MIMIC-III paper in any publications: Johnson et al., *MIMIC-III, a freely accessible critical care database*, Scientific Data, 2016.
>
> For full details, see the [PhysioNet Credentialing Guide](https://physionet.org/settings/credentialing/).

### Alternative Datasets

If you don't have access to MIMIC-III, you can still follow this tutorial using alternative datasets:

1. **eICU Collaborative Research Database** (also on BigQuery)
   - Available at [PhysioNet](https://physionet.org/content/eicu-crd/2.0/) with the same credentialing process
   - Multi-center ICU dataset with similar table structure (patient, vital signs, labs)
   - BigQuery dataset: `physionet-data.eicu_crd`
   - Adjust table/column names but the pipeline logic remains the same

2. **Kaggle Healthcare Datasets**
   - [WiDS Datathon ICU Dataset](https://www.kaggle.com/c/widsdatathon2020/) -- pre-processed ICU mortality data
   - Search Kaggle for "ICU mortality" for additional options
   - These are typically pre-merged CSV files, so you can skip the BigQuery loading step and load directly with `pd.read_csv()`

3. **Synthetic Data**
   - For learning purposes, you can generate synthetic ICU data using libraries like `faker` or `sdv`
   - This won't produce clinically meaningful results but lets you practice the ML pipeline

If using an alternative dataset, modify the `load_mimic_tables` function or replace it with a simple CSV loader.

In [ ]:
import pandas as pd
from google.cloud import bigquery
from google.colab import auth


def load_mimic_tables(
    project_id: str,
    dataset: str = "physionet-data.mimiciii_clinical",
) -> dict:
    """Authenticate with Google Cloud, create a BigQuery client, and query
    the five core MIMIC-III tables needed for ICU mortality prediction.

    Parameters
    ----------
    project_id : str
        Your Google Cloud project ID (used for billing BigQuery queries).
    dataset : str
        Fully-qualified BigQuery dataset path for MIMIC-III.

    Returns
    -------
    dict[str, pd.DataFrame]
        Mapping of table name to its DataFrame.
    """

    # ------------------------------------------------------------------
    # Step 1: Authenticate with Google Cloud
    # ------------------------------------------------------------------
    # In Colab, this pops up an OAuth consent screen. You must be logged
    # into a Google account that has accepted the PhysioNet data use
    # agreement and has BigQuery access to physionet-data.
    auth.authenticate_user()
    print("Google Cloud authentication successful.")

    # ------------------------------------------------------------------
    # Step 2: Create a BigQuery client
    # ------------------------------------------------------------------
    # The project_id determines which GCP project is billed for queries.
    # MIMIC-III on BigQuery is free to query, but you still need a project.
    client = bigquery.Client(project=project_id)
    print(f"BigQuery client created for project: {project_id}")

    # ------------------------------------------------------------------
    # Step 3: Define SQL queries for each table
    # ------------------------------------------------------------------
    # We query only the columns we need to reduce data transfer.
    # For chartevents and labevents, we filter by ITEMID in the SQL
    # query itself -- these tables are enormous (>300M rows for
    # chartevents), so filtering in SQL is essential for performance.
    # ------------------------------------------------------------------

    # Key vital sign ITEMIDs (MetaVision system in MIMIC-III)
    # These map to the most commonly used bedside vital signs
    vital_itemids = [
        220045,  # Heart Rate
        220050,  # Arterial Blood Pressure systolic
        220051,  # Arterial Blood Pressure diastolic
        220052,  # Arterial Blood Pressure mean
        220179,  # Non-Invasive Blood Pressure systolic
        220180,  # Non-Invasive Blood Pressure diastolic
        220181,  # Non-Invasive Blood Pressure mean
        220210,  # Respiratory Rate
        220277,  # SpO2 (oxygen saturation)
        223761,  # Temperature Fahrenheit
        223762,  # Temperature Celsius
    ]

    # Key lab ITEMIDs -- common labs used in severity-of-illness scores
    lab_itemids = [
        51301,  # White Blood Cell count
        51222,  # Hemoglobin
        51265,  # Platelet count
        50983,  # Sodium
        50971,  # Potassium
        50882,  # Bicarbonate
        51006,  # Blood Urea Nitrogen (BUN)
        50912,  # Creatinine
        50931,  # Glucose
        50813,  # Lactate
    ]

    # Convert ITEMID lists to comma-separated strings for SQL IN clause
    vital_ids_str = ", ".join(str(i) for i in vital_itemids)
    lab_ids_str = ", ".join(str(i) for i in lab_itemids)

    queries = {
        # PATIENTS: basic demographics
        "patients": f"""
            SELECT subject_id, gender, dob
            FROM `{dataset}.patients`
        """,

        # ADMISSIONS: hospital admission info + mortality label
        "admissions": f"""
            SELECT subject_id, hadm_id, admittime, dischtime,
                   admission_type, insurance, hospital_expire_flag
            FROM `{dataset}.admissions`
        """,

        # ICUSTAYS: ICU stay timestamps (defines our prediction window)
        "icustays": f"""
            SELECT subject_id, hadm_id, icustay_id, intime, outtime
            FROM `{dataset}.icustays`
        """,

        # CHARTEVENTS: vital signs, filtered to key ITEMIDs only
        # Without the ITEMID filter, this query would scan 300M+ rows
        "chartevents": f"""
            SELECT subject_id, hadm_id, itemid, charttime, valuenum
            FROM `{dataset}.chartevents`
            WHERE itemid IN ({vital_ids_str})
              AND valuenum IS NOT NULL
        """,

        # LABEVENTS: lab results, filtered to key ITEMIDs only
        "labevents": f"""
            SELECT subject_id, hadm_id, itemid, charttime, valuenum
            FROM `{dataset}.labevents`
            WHERE itemid IN ({lab_ids_str})
              AND valuenum IS NOT NULL
        """,
    }

    # ------------------------------------------------------------------
    # Step 4: Execute queries and collect results
    # ------------------------------------------------------------------
    tables = {}
    for table_name, sql in queries.items():
        print(f"Querying {table_name}...")
        tables[table_name] = client.query(sql).to_dataframe()
        print(f"  -> {len(tables[table_name]):,} rows loaded.")

    print("\nAll MIMIC-III tables loaded successfully!")
    return tables

In [ ]:
def merge_icu_data(tables: dict) -> pd.DataFrame:
    """Merge MIMIC-III tables into a single DataFrame for ICU mortality prediction.

    This function:
    1. Joins patients + admissions + icustays on subject_id / hadm_id
    2. Computes patient age from date of birth and admission time
    3. Filters chartevents and labevents to the first 24 hours of ICU stay
    4. Pivots vital signs and labs into per-stay columns
    5. Creates the binary mortality label from hospital_expire_flag

    Parameters
    ----------
    tables : dict[str, pd.DataFrame]
        Output from load_mimic_tables().

    Returns
    -------
    pd.DataFrame
        Merged DataFrame with one row per ICU stay, containing demographics,
        first-24h vital signs, first-24h lab values, and the mortality label.
    """

    # ------------------------------------------------------------------
    # Step 1: Merge patients + admissions + icustays
    # ------------------------------------------------------------------
    # We join on subject_id first (patient-level), then on hadm_id
    # (admission-level). This gives us one row per ICU stay with
    # demographics, admission info, and ICU timestamps.
    patients = tables["patients"].copy()
    admissions = tables["admissions"].copy()
    icustays = tables["icustays"].copy()

    # Merge patients with admissions on subject_id
    merged = pd.merge(patients, admissions, on="subject_id", how="inner")

    # Merge with icustays on both subject_id and hadm_id
    merged = pd.merge(merged, icustays, on=["subject_id", "hadm_id"], how="inner")

    print(f"After merging patients + admissions + icustays: {len(merged):,} ICU stays")

    # ------------------------------------------------------------------
    # Step 2: Compute age
    # ------------------------------------------------------------------
    # MIMIC-III stores DOB as a datetime. For patients >89 years old,
    # the DOB is shifted to 300 years before admission (for de-identification).
    # We compute age in years and cap at 90 for those shifted records.
    merged["admittime"] = pd.to_datetime(merged["admittime"])
    merged["dob"] = pd.to_datetime(merged["dob"])
    merged["intime"] = pd.to_datetime(merged["intime"])

    # Age in years = difference between admission and birth, converted to days then years
    merged["age"] = (merged["admittime"] - merged["dob"]).dt.days / 365.25

    # Cap age at 90 for de-identified elderly patients (MIMIC convention)
    merged.loc[merged["age"] > 200, "age"] = 90.0

    # Remove neonates (age < 18) -- this tutorial focuses on adult ICU patients
    merged = merged[merged["age"] >= 18].copy()
    print(f"After filtering to adults (age >= 18): {len(merged):,} ICU stays")

    # ------------------------------------------------------------------
    # Step 3: Filter chartevents to first 24h of ICU stay
    # ------------------------------------------------------------------
    # The first 24 hours is the standard prediction window in ICU
    # mortality research. Data collected after 24h would introduce
    # look-ahead bias into our predictions.
    chartevents = tables["chartevents"].copy()
    chartevents["charttime"] = pd.to_datetime(chartevents["charttime"])

    # Join chartevents with ICU stay info to get intime for each event
    charts_with_icu = pd.merge(
        chartevents,
        merged[["subject_id", "hadm_id", "icustay_id", "intime"]],
        on=["subject_id", "hadm_id"],
        how="inner",
    )

    # Keep only events within the first 24 hours of ICU admission
    charts_24h = charts_with_icu[
        (charts_with_icu["charttime"] >= charts_with_icu["intime"])
        & (charts_with_icu["charttime"] <= charts_with_icu["intime"] + pd.Timedelta(hours=24))
    ].copy()
    print(f"Chartevents in first 24h: {len(charts_24h):,} measurements")

    # ------------------------------------------------------------------
    # Step 4: Filter labevents to first 24h of ICU stay
    # ------------------------------------------------------------------
    labevents = tables["labevents"].copy()
    labevents["charttime"] = pd.to_datetime(labevents["charttime"])

    labs_with_icu = pd.merge(
        labevents,
        merged[["subject_id", "hadm_id", "icustay_id", "intime"]],
        on=["subject_id", "hadm_id"],
        how="inner",
    )

    labs_24h = labs_with_icu[
        (labs_with_icu["charttime"] >= labs_with_icu["intime"])
        & (labs_with_icu["charttime"] <= labs_with_icu["intime"] + pd.Timedelta(hours=24))
    ].copy()
    print(f"Labevents in first 24h: {len(labs_24h):,} measurements")

    # ------------------------------------------------------------------
    # Step 5: Pivot vitals and labs into per-stay columns
    # ------------------------------------------------------------------
    # We create a mapping from ITEMID to a human-readable column name.
    # Then we pivot so each vital/lab becomes its own column with the
    # mean value over the first 24h (aggregation happens in later sections).

    vital_names = {
        220045: "heart_rate", 220050: "sbp_arterial", 220051: "dbp_arterial",
        220052: "map_arterial", 220179: "sbp_noninvasive", 220180: "dbp_noninvasive",
        220181: "map_noninvasive", 220210: "resp_rate", 220277: "spo2",
        223761: "temp_f", 223762: "temp_c",
    }

    lab_names = {
        51301: "wbc", 51222: "hemoglobin", 51265: "platelet",
        50983: "sodium", 50971: "potassium", 50882: "bicarbonate",
        51006: "bun", 50912: "creatinine", 50931: "glucose", 50813: "lactate",
    }

    # Map ITEMIDs to readable names
    charts_24h["vital_name"] = charts_24h["itemid"].map(vital_names)
    labs_24h["lab_name"] = labs_24h["itemid"].map(lab_names)

    # Pivot vitals: one column per vital sign, mean value per ICU stay
    vitals_pivot = charts_24h.pivot_table(
        index="icustay_id", columns="vital_name", values="valuenum", aggfunc="mean"
    ).reset_index()

    # Pivot labs: one column per lab test, mean value per ICU stay
    labs_pivot = labs_24h.pivot_table(
        index="icustay_id", columns="lab_name", values="valuenum", aggfunc="mean"
    ).reset_index()

    # ------------------------------------------------------------------
    # Step 6: Merge pivoted vitals and labs back into the main DataFrame
    # ------------------------------------------------------------------
    result = pd.merge(merged, vitals_pivot, on="icustay_id", how="left")
    result = pd.merge(result, labs_pivot, on="icustay_id", how="left")

    # Create the binary mortality label
    result["mortality"] = result["hospital_expire_flag"].astype(int)

    print(f"\nFinal merged dataset: {len(result):,} ICU stays, {result.shape[1]} columns")
    return result

In [ ]:
def display_data_summary(df: pd.DataFrame) -> None:
    """Print a comprehensive summary of the merged ICU dataset.

    Displays:
    - Total number of unique patients and ICU stays
    - Mortality rate (percentage of patients who died in hospital)
    - Number of features (columns)
    - Data types breakdown
    - Missingness statistics per column

    Parameters
    ----------
    df : pd.DataFrame
        The merged ICU DataFrame from merge_icu_data().
    """

    # ------------------------------------------------------------------
    # Patient and stay counts
    # ------------------------------------------------------------------
    n_patients = df["subject_id"].nunique()
    n_stays = len(df)
    mortality_rate = df["mortality"].mean() * 100

    print("=" * 60)
    print("DATASET SUMMARY")
    print("=" * 60)
    print(f"Total unique patients:  {n_patients:,}")
    print(f"Total ICU stays:        {n_stays:,}")
    print(f"Mortality rate:         {mortality_rate:.1f}%")
    print(f"Number of features:     {df.shape[1]}")

    # ------------------------------------------------------------------
    # Data types breakdown
    # ------------------------------------------------------------------
    # Understanding data types helps plan preprocessing:
    # - numeric columns need imputation and normalization
    # - object/categorical columns need encoding
    # - datetime columns are used for filtering, not as features
    print("\n" + "-" * 60)
    print("DATA TYPES")
    print("-" * 60)
    dtype_counts = df.dtypes.value_counts()
    for dtype, count in dtype_counts.items():
        print(f"  {str(dtype):.<30} {count} columns")

    # ------------------------------------------------------------------
    # Missingness statistics
    # ------------------------------------------------------------------
    # Missing data is common in clinical datasets -- not every patient
    # has every lab drawn or every vital recorded. Understanding the
    # missingness pattern is critical before imputation.
    print("\n" + "-" * 60)
    print("MISSINGNESS SUMMARY")
    print("-" * 60)

    missing = df.isnull().sum()
    missing_pct = (missing / len(df) * 100).round(1)

    # Create a summary DataFrame sorted by missingness (descending)
    miss_df = pd.DataFrame({
        "Missing Count": missing,
        "Missing %": missing_pct,
    }).sort_values("Missing %", ascending=False)

    # Only show columns that have some missing data
    miss_df = miss_df[miss_df["Missing Count"] > 0]

    if len(miss_df) == 0:
        print("  No missing values found!")
    else:
        print(f"  {len(miss_df)} columns have missing values:\n")
        print(miss_df.to_string())

    print("\n" + "=" * 60)

### Load Data from BigQuery

Now let's run the data loading pipeline. This will:
1. Authenticate with Google Cloud (you'll see an OAuth popup)
2. Query each MIMIC-III table from BigQuery
3. Merge tables and filter to the first 24 hours of each ICU stay
4. Display a summary of the resulting dataset

**Note:** Replace `"your-gcp-project-id"` below with your actual Google Cloud project ID.

In [ ]:
# ============================================================
# Execute the data loading pipeline
# ============================================================
# Replace the project_id with your own GCP project ID.
# The dataset parameter defaults to physionet-data.mimiciii_clinical
# which is the standard BigQuery location for MIMIC-III.
# ============================================================

# Set your GCP project ID here
PROJECT_ID = "your-gcp-project-id"  # <-- CHANGE THIS to your project ID

# Load all five MIMIC-III tables from BigQuery
tables = load_mimic_tables(project_id=PROJECT_ID)

# Merge tables into a single analysis-ready DataFrame
icu_data = merge_icu_data(tables)

# Display a comprehensive summary of the dataset
display_data_summary(icu_data)

In [ ]:
# ============================================================
# Preview the merged dataset
# ============================================================
# Let's look at the first few rows to verify the merge worked
# correctly. You should see demographics, vital signs, labs,
# and the mortality label all in one DataFrame.
# ============================================================

# Show the first 5 rows -- check that columns look reasonable
icu_data.head()

---

<a id="section-3-preprocessing--cleaning"></a>

## 3. Preprocessing & Cleaning

### Learning ObjectivesBy the end of this section, you will:- Understand why **missing data** is common in clinical datasets and how to handle it appropriately- Apply **median imputation** for laboratory values and vital signs aggregated to per-stay means- Detect and clip **physiologically implausible outliers** using clinically validated ranges- Remove features with excessive missingness (>80%) that would add noise rather than signal- **Normalize** continuous features to zero mean and unit variance for model compatibility- Compare before-and-after missingness statistics to verify preprocessing worked correctly- Appreciate the **clinical rationale** behind each preprocessing decision

### Preprocessing OverviewClinical data from the ICU is inherently messy. Not every patient has every lab drawn, vital signs may be recorded at irregular intervals, and measurement errors can produce physiologically impossible values. Our preprocessing pipeline addresses these issues in four steps:| Step | Function | What It Does | Clinical Rationale ||------|----------|-------------|--------------------|| 1 | `drop_high_missing()` | Drop features with >80% missing values | Features missing for most patients carry little predictive signal and can introduce noise || 2 | `handle_outliers()` | Clip values to clinically valid ranges | Physiologically impossible values (e.g., heart rate of 5000) are measurement errors, not real data || 3 | `impute_missing()` | Fill remaining NaN values with column medians | Median is robust to outliers; appropriate for per-stay aggregated values || 4 | `normalize_features()` | Standardize to zero mean, unit variance | Many ML algorithms (logistic regression, neural networks) perform better with standardized inputs |We apply these steps in this order because: (1) dropping high-missing features first avoids wasting effort imputing columns we'll discard, (2) clipping outliers before imputation prevents extreme values from skewing the median, (3) imputation fills gaps so normalization can operate on complete data.

In [ ]:
import warnings
from sklearn.preprocessing import StandardScaler


def drop_high_missing(df: pd.DataFrame, threshold: float = 0.8) -> pd.DataFrame:
    """Drop features where the fraction of missing values exceeds the threshold.

    Clinical rationale: Features that are missing for >80% of patients provide
    very little predictive information. Including them would force heavy imputation
    that introduces artificial patterns rather than real clinical signal.

    Parameters
    ----------
    df : pd.DataFrame
        Input DataFrame (typically the merged ICU data).
    threshold : float
        Maximum allowed fraction of missing values (default 0.8 = 80%).
        Features exceeding this threshold are dropped.

    Returns
    -------
    pd.DataFrame
        DataFrame with high-missingness features removed.
    """
    # Calculate the fraction of missing values for each column
    missing_frac = df.isnull().mean()

    # Identify columns that exceed the missingness threshold
    cols_to_drop = missing_frac[missing_frac > threshold].index.tolist()

    # Log a warning for each dropped column so the user knows what was removed
    # and can investigate if a clinically important feature was unexpectedly dropped
    for col in cols_to_drop:
        pct = missing_frac[col] * 100
        warnings.warn(
            f"Dropping '{col}': {pct:.1f}% missing (threshold: {threshold * 100:.0f}%)"
        )

    # Drop the identified columns
    df_cleaned = df.drop(columns=cols_to_drop)

    print(f"Dropped {len(cols_to_drop)} features with >{threshold*100:.0f}% missing values.")
    if cols_to_drop:
        print(f"  Dropped columns: {cols_to_drop}")
    print(f"  Remaining features: {df_cleaned.shape[1]}")

    return df_cleaned

In [ ]:
def handle_outliers(df: pd.DataFrame, clinical_ranges: dict) -> pd.DataFrame:
    """Clip values outside clinically valid ranges to remove measurement errors.

    Clinical rationale: ICU monitoring equipment and manual data entry can produce
    physiologically impossible values (e.g., a heart rate of 5000 bpm or a negative
    SpO2). These are not real clinical observations -- they are artifacts that would
    distort our models. We clip to ranges established by clinical domain knowledge.

    Parameters
    ----------
    df : pd.DataFrame
        Input DataFrame with numeric clinical features.
    clinical_ranges : dict[str, tuple[float, float]]
        Mapping of column name to (min_valid, max_valid) range.
        Example: {"heart_rate": (0, 300), "spo2": (0, 100)}

    Returns
    -------
    pd.DataFrame
        DataFrame with out-of-range values clipped to the valid bounds.
    """
    df_clipped = df.copy()
    clipped_count = 0

    for col, (low, high) in clinical_ranges.items():
        if col not in df_clipped.columns:
            # Skip columns not present in the DataFrame (e.g., if they were
            # dropped in a previous step). Log an info message for transparency.
            print(f"  Info: '{col}' not in DataFrame, skipping outlier clipping.")
            continue

        # Count how many values fall outside the valid range before clipping
        # This helps us understand the extent of data quality issues
        n_below = (df_clipped[col] < low).sum()
        n_above = (df_clipped[col] > high).sum()
        n_outliers = n_below + n_above

        if n_outliers > 0:
            clipped_count += n_outliers
            print(f"  {col}: clipped {n_outliers} values to [{low}, {high}] "
                  f"({n_below} below, {n_above} above)")

        # Clip values to the valid range using pandas clip()
        # Values below 'low' become 'low'; values above 'high' become 'high'
        df_clipped[col] = df_clipped[col].clip(lower=low, upper=high)

    print(f"\nTotal outlier values clipped: {clipped_count:,}")
    return df_clipped

In [ ]:
def impute_missing(df: pd.DataFrame, strategy_map: dict) -> pd.DataFrame:
    """Impute missing values using the specified strategy for each column group.

    Clinical rationale: Missing lab and vital sign values in ICU data usually mean
    the test was not ordered or the measurement was not recorded -- not that the
    value is zero. Median imputation is appropriate here because:
    - Our data is already aggregated to per-stay means (one row per ICU stay),
      so forward-fill across time is not applicable
    - Median is robust to the skewed distributions common in clinical data
      (e.g., lactate and creatinine are right-skewed)
    - It preserves the central tendency without being pulled by extreme values

    Parameters
    ----------
    df : pd.DataFrame
        Input DataFrame with missing values.
    strategy_map : dict[str, str]
        Mapping of column name to imputation strategy.
        Supported strategies: "median" (fill with column median).

    Returns
    -------
    pd.DataFrame
        DataFrame with missing values imputed.
    """
    df_imputed = df.copy()
    imputed_counts = {}

    for col, strategy in strategy_map.items():
        if col not in df_imputed.columns:
            # Column may have been dropped in a previous step
            continue

        n_missing = df_imputed[col].isnull().sum()
        if n_missing == 0:
            continue  # No missing values to impute

        if strategy == "median":
            # Compute the median of non-missing values
            median_val = df_imputed[col].median()
            df_imputed[col] = df_imputed[col].fillna(median_val)
            imputed_counts[col] = (n_missing, f"median={median_val:.2f}")
        else:
            warnings.warn(f"Unknown imputation strategy '{strategy}' for column '{col}'")

    # Print a summary of what was imputed
    print(f"Imputed {len(imputed_counts)} columns:")
    for col, (count, detail) in imputed_counts.items():
        print(f"  {col}: {count} values filled ({detail})")

    return df_imputed

In [ ]:
def normalize_features(
    df: pd.DataFrame, feature_cols: list
) -> tuple:
    """Standardize continuous features to zero mean and unit variance.

    Clinical rationale: Different clinical measurements have vastly different
    scales -- heart rate is typically 60-100, while WBC count might be 4-11
    (x10^3/uL). Without normalization:
    - Logistic regression coefficients would be dominated by large-scale features
    - Neural network gradient descent would be inefficient
    - Distance-based methods would be biased toward high-magnitude features

    We use StandardScaler (z-score normalization) which transforms each feature
    to have mean=0 and std=1. The fitted scaler is returned so we can apply the
    same transformation to validation and test sets later (preventing data leakage).

    Parameters
    ----------
    df : pd.DataFrame
        Input DataFrame containing the features to normalize.
    feature_cols : list[str]
        List of column names to normalize (numeric features only).

    Returns
    -------
    tuple[pd.DataFrame, StandardScaler]
        - DataFrame with normalized feature columns (other columns unchanged)
        - Fitted StandardScaler object (save this for transforming test data)
    """
    df_normalized = df.copy()

    # Initialize the scaler -- it will learn mean and std from the data
    scaler = StandardScaler()

    # Fit the scaler on the feature columns and transform them
    # fit_transform() computes mean/std and applies the transformation in one step
    # Only columns in feature_cols are affected; IDs, labels, etc. remain unchanged
    df_normalized[feature_cols] = scaler.fit_transform(df_normalized[feature_cols])

    print(f"Normalized {len(feature_cols)} features to zero mean, unit variance.")
    print(f"  Scaler fitted -- save this to transform test data later!")

    return df_normalized, scaler

### Step 1: Examine Missingness Before PreprocessingBefore applying any preprocessing, let's examine the current state of our data. Understanding the missingness pattern helps us:- Identify which features might need to be dropped entirely- Understand which clinical measurements are commonly missing (e.g., arterial blood pressure requires an arterial line, so it's often missing for less-sick patients)- Set a baseline to compare against after preprocessing

In [ ]:
# ============================================================
# Display missingness statistics BEFORE preprocessing
# ============================================================
# This gives us a baseline to compare against after we apply
# imputation. In clinical data, missingness is informative:
# - Labs like lactate are only drawn when sepsis is suspected
# - Arterial BP requires an invasive line (sicker patients)
# - Some vitals (e.g., temp) may not be recorded every hour
# ============================================================

print("=" * 60)
print("MISSINGNESS BEFORE PREPROCESSING")
print("=" * 60)

# Calculate missingness for each column
missing_before = icu_data.isnull().sum()
missing_pct_before = (missing_before / len(icu_data) * 100).round(1)

# Build a summary DataFrame for display
missingness_before = pd.DataFrame({
    "Missing Count": missing_before,
    "Missing %": missing_pct_before,
    "Present %": (100 - missing_pct_before).round(1)
}).sort_values("Missing %", ascending=False)

# Show only columns with missing data
missingness_before_nonzero = missingness_before[missingness_before["Missing Count"] > 0]
print(f"\n{len(missingness_before_nonzero)} of {len(missingness_before)} columns have missing values:\n")
print(missingness_before_nonzero.to_string())
print("\n" + "=" * 60)

### Step 2: Apply the Preprocessing PipelineNow we apply our four preprocessing steps in order. We define the clinical ranges for outlier clipping and the imputation strategies, then run each function sequentially.**Important:** We save a copy of the data before preprocessing so we can compare before-and-after statistics at the end.

In [ ]:
# ============================================================
# Apply the full preprocessing pipeline
# ============================================================
# We process the data in a specific order:
# 1. Drop high-missing features (>80% missing)
# 2. Clip outliers to clinically valid ranges
# 3. Impute remaining missing values with median
# 4. Normalize features to zero mean, unit variance
#
# Each step builds on the previous one. The order matters!
# ============================================================

# Save a copy for before/after comparison
icu_data_raw = icu_data.copy()

# ------------------------------------------------------------------
# Step 1: Drop features with >80% missing values
# ------------------------------------------------------------------
# Features missing for most patients add noise, not signal.
# The 80% threshold is a common choice in clinical ML literature.
print("STEP 1: Dropping high-missingness features")
print("-" * 50)
icu_data = drop_high_missing(icu_data, threshold=0.8)

# ------------------------------------------------------------------
# Step 2: Clip outliers to clinically valid ranges
# ------------------------------------------------------------------
# These ranges are based on physiological limits. Values outside
# these bounds are almost certainly measurement errors or data
# entry mistakes, not real patient observations.
print("\nSTEP 2: Clipping outliers to clinical ranges")
print("-" * 50)

# Define clinically valid ranges for each measurement
# Ranges are intentionally wide to avoid removing real extreme values
# that occur in critically ill patients (e.g., HR of 200 in SVT)
clinical_ranges = {
    # Vital signs
    "heart_rate": (0, 300),          # Normal: 60-100, but tachyarrhythmias can exceed 200
    "sbp_arterial": (0, 300),        # Normal: 90-140, but hypertensive crises can reach 250+
    "dbp_arterial": (0, 200),        # Normal: 60-90
    "map_arterial": (0, 250),        # Normal: 70-105, MAP = (SBP + 2*DBP) / 3
    "sbp_noninvasive": (0, 300),     # Same ranges as arterial (different measurement method)
    "dbp_noninvasive": (0, 200),
    "map_noninvasive": (0, 250),
    "resp_rate": (0, 70),            # Normal: 12-20, but can be 40+ in respiratory distress
    "spo2": (0, 100),               # Oxygen saturation is a percentage, cannot exceed 100%
    "temp_f": (90, 115),            # Normal: 97-99F, hypothermia/hyperthermia extremes
    "temp_c": (32, 45),             # Normal: 36-37.5C, equivalent to temp_f range

    # Laboratory values
    "wbc": (0, 500),                # Normal: 4-11, but leukemoid reactions can be very high
    "hemoglobin": (0, 25),          # Normal: 12-17, polycythemia can push higher
    "platelet": (0, 2000),          # Normal: 150-400, reactive thrombocytosis can be very high
    "sodium": (100, 180),           # Normal: 135-145, severe dysnatremias are life-threatening
    "potassium": (1, 12),           # Normal: 3.5-5.0, extremes cause cardiac arrest
    "bicarbonate": (0, 60),         # Normal: 22-28, metabolic alkalosis can push higher
    "bun": (0, 300),                # Normal: 7-20, renal failure can cause very high BUN
    "creatinine": (0, 30),          # Normal: 0.7-1.3, severe renal failure can be very high
    "glucose": (0, 1500),           # Normal: 70-100, DKA can cause extreme hyperglycemia
    "lactate": (0, 30),             # Normal: 0.5-2.0, severe sepsis/shock can be very high
}

icu_data = handle_outliers(icu_data, clinical_ranges)

In [ ]:
# ------------------------------------------------------------------
# Step 3: Impute remaining missing values
# ------------------------------------------------------------------
# We use median imputation for all clinical features (both labs and
# vitals). Since our data is already aggregated to per-stay means
# (one value per ICU stay per feature), forward-fill across time
# does not apply here. Median is preferred over mean because:
# - It is robust to the skewed distributions common in clinical data
# - It is not influenced by the outliers we just clipped
# - It represents a "typical" patient value
print("\nSTEP 3: Imputing missing values")
print("-" * 50)

# Define which columns to impute and their strategy
# All clinical measurements use median imputation
vital_cols = [
    "heart_rate", "sbp_arterial", "dbp_arterial", "map_arterial",
    "sbp_noninvasive", "dbp_noninvasive", "map_noninvasive",
    "resp_rate", "spo2", "temp_f", "temp_c",
]

lab_cols = [
    "wbc", "hemoglobin", "platelet", "sodium", "potassium",
    "bicarbonate", "bun", "creatinine", "glucose", "lactate",
]

# Build the strategy map: all columns use median imputation
# (since data is already aggregated to per-stay means)
strategy_map = {}
for col in vital_cols + lab_cols:
    strategy_map[col] = "median"

icu_data = impute_missing(icu_data, strategy_map)

In [ ]:
# ------------------------------------------------------------------
# Step 4: Normalize continuous features
# ------------------------------------------------------------------
# Standardize numeric features to zero mean and unit variance.
# We normalize ONLY the clinical measurement columns -- not IDs,
# not the mortality label, and not categorical columns.
#
# IMPORTANT: We save the fitted scaler so we can apply the SAME
# transformation to validation/test data later. Fitting a new
# scaler on test data would be data leakage!
print("\nSTEP 4: Normalizing features")
print("-" * 50)

# Identify which columns to normalize: all numeric clinical features
# that are still present after the previous steps
numeric_feature_cols = [
    col for col in vital_cols + lab_cols + ["age"]
    if col in icu_data.columns
]

icu_data, feature_scaler = normalize_features(icu_data, numeric_feature_cols)

print("\nPreprocessing pipeline complete!")

### Step 3: Verify Preprocessing — Before vs. After ComparisonLet's compare the missingness statistics before and after preprocessing to confirm that our pipeline worked correctly. After preprocessing, we expect:- **Zero missing values** in all imputed clinical columns- **Fewer columns** if any were dropped for high missingness- **All values within clinical ranges** (no physiologically impossible outliers)

In [ ]:
# ============================================================
# Display missingness statistics AFTER preprocessing
# ============================================================
# Compare before and after to verify our preprocessing pipeline
# successfully handled missing values, outliers, and normalization.
# ============================================================

print("=" * 60)
print("MISSINGNESS AFTER PREPROCESSING")
print("=" * 60)

# Calculate missingness after preprocessing
missing_after = icu_data.isnull().sum()
missing_pct_after = (missing_after / len(icu_data) * 100).round(1)

missingness_after = pd.DataFrame({
    "Missing Count": missing_after,
    "Missing %": missing_pct_after,
}).sort_values("Missing %", ascending=False)

missingness_after_nonzero = missingness_after[missingness_after["Missing Count"] > 0]

if len(missingness_after_nonzero) == 0:
    print("\nNo missing values remain in any column!")
else:
    print(f"\n{len(missingness_after_nonzero)} columns still have missing values:\n")
    print(missingness_after_nonzero.to_string())

# ------------------------------------------------------------------
# Before vs. After comparison for clinical columns
# ------------------------------------------------------------------
print("\n" + "=" * 60)
print("BEFORE vs. AFTER COMPARISON (Clinical Columns)")
print("=" * 60)

# Compare only the clinical columns that existed in both versions
clinical_cols = [c for c in vital_cols + lab_cols if c in icu_data_raw.columns]
comparison_data = []

for col in clinical_cols:
    before_missing = icu_data_raw[col].isnull().sum()
    before_pct = (before_missing / len(icu_data_raw) * 100)

    if col in icu_data.columns:
        after_missing = icu_data[col].isnull().sum()
        after_pct = (after_missing / len(icu_data) * 100)
        status = "Imputed" if before_missing > 0 and after_missing == 0 else "OK"
    else:
        after_missing = "DROPPED"
        after_pct = "N/A"
        status = "Dropped (high missing)"

    comparison_data.append({
        "Feature": col,
        "Before Missing": before_missing,
        "Before %": f"{before_pct:.1f}%",
        "After Missing": after_missing,
        "Status": status,
    })

comparison_df = pd.DataFrame(comparison_data)
print("\n" + comparison_df.to_string(index=False))

print("\n" + "=" * 60)
print("Preprocessing complete! Data is ready for feature engineering.")
print("=" * 60)

---

<a id="section-4-feature-engineering"></a>

## 4. Feature Engineering

### Learning Objectives

By the end of this section, you will:

- Encode categorical variables using one-hot encoding
- Build a final feature matrix and label vector for model training
- Visualize feature correlations with a heatmap
- Split data into train/validation/test sets with stratified sampling

In [ ]:
from sklearn.model_selection import train_test_split
import seaborn as sns
import matplotlib.pyplot as plt

# ============================================================
# One-hot encode categorical variables
# ============================================================
# Gender, admission type, and insurance are categorical.
# One-hot encoding creates binary indicator columns for each
# category, which ML models can process as numeric features.
# ============================================================

def encode_categoricals(df, cat_cols):
    """One-hot encode categorical columns.
    Returns DataFrame with binary indicator columns."""
    return pd.get_dummies(df, columns=cat_cols, drop_first=False, dtype=int)

def build_feature_matrix(df, feature_cols, label_col='mortality'):
    """Extract feature matrix X and label vector y as numpy arrays."""
    X = df[feature_cols].values.astype(np.float32)
    y = df[label_col].values.astype(np.float32)
    return X, y

# Identify categorical columns present in our data
cat_cols = [c for c in ['gender', 'admission_type', 'insurance'] if c in icu_data.columns]
print(f'Categorical columns to encode: {cat_cols}')

# Apply one-hot encoding
icu_encoded = encode_categoricals(icu_data, cat_cols)
print(f'After encoding: {icu_encoded.shape[1]} columns')

In [ ]:
# ============================================================
# Build the feature matrix
# ============================================================
# Select all numeric feature columns (vitals, labs, age, encoded cats)
# Exclude ID columns, timestamps, and the label itself.
# ============================================================

# Columns to exclude from features
exclude_cols = ['subject_id', 'hadm_id', 'icustay_id', 'admittime',
                'dischtime', 'dob', 'intime', 'outtime',
                'hospital_expire_flag', 'mortality']

# Feature columns = all numeric columns not in exclude list
feature_cols = [c for c in icu_encoded.columns
                if c not in exclude_cols
                and icu_encoded[c].dtype in ['float64','float32','int64','int32','uint8']]

print(f'Number of features: {len(feature_cols)}')
print(f'Feature columns: {feature_cols}')

# Build X and y
X, y = build_feature_matrix(icu_encoded, feature_cols)
feature_names = feature_cols  # Save for SHAP later

print(f'\nFeature matrix X shape: {X.shape}')
print(f'Label vector y shape: {y.shape}')
print(f'Mortality rate: {y.mean()*100:.1f}%')
print(f'Class distribution: {int(y.sum())} died / {int(len(y)-y.sum())} survived')

In [ ]:
# ============================================================
# Correlation heatmap of top features vs mortality
# ============================================================
# Visualize which features are most correlated with mortality.
# This gives us clinical intuition before training models.
# ============================================================

# Compute correlations with mortality
corr_with_mortality = icu_encoded[feature_cols + ['mortality']].corr()['mortality'].drop('mortality')
top_features = corr_with_mortality.abs().sort_values(ascending=False).head(15).index.tolist()

# Plot heatmap of top features
plt.figure(figsize=(10, 8))
corr_matrix = icu_encoded[top_features + ['mortality']].corr()
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='RdBu_r', center=0,
            square=True, linewidths=0.5)
plt.title('Correlation Heatmap: Top 15 Features vs Mortality')
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# Train / Validation / Test split (70/15/15)
# ============================================================
# We use stratified splitting to preserve the mortality rate
# in each subset. This is critical because mortality is rare
# (~11-13%), and random splits could create subsets with very
# different class distributions.
# ============================================================

# First split: 70% train, 30% temp
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.3, random_state=SEED, stratify=y
)

# Second split: 50/50 of temp -> 15% val, 15% test
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=SEED, stratify=y_temp
)

print(f'Train: {X_train.shape[0]} samples ({y_train.mean()*100:.1f}% mortality)')
print(f'Val:   {X_val.shape[0]} samples ({y_val.mean()*100:.1f}% mortality)')
print(f'Test:  {X_test.shape[0]} samples ({y_test.mean()*100:.1f}% mortality)')
print(f'\nStratification preserved: all subsets have similar mortality rates.')

---

<a id="section-5-traditional-ml-models"></a>

## 5. Traditional ML Models

### Learning Objectives

By the end of this section, you will:

- Train three traditional ML models: Logistic Regression, Random Forest, and XGBoost
- Use stratified k-fold cross-validation for robust performance estimation
- Handle class imbalance using class weighting and scale_pos_weight
- Compare cross-validation AUROC scores across models

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import roc_auc_score
import xgboost as xgb

# ============================================================
# Traditional ML training functions
# ============================================================

def train_logistic_regression(X, y, cv=5):
    """Train Logistic Regression with stratified k-fold CV.
    Uses class_weight='balanced' to handle mortality imbalance."""
    model = LogisticRegression(
        max_iter=1000,
        class_weight='balanced',  # Upweight minority class
        random_state=SEED,
        solver='lbfgs'
    )
    skf = StratifiedKFold(n_splits=cv, shuffle=True, random_state=SEED)
    scores = cross_val_score(model, X, y, cv=skf, scoring='roc_auc')
    model.fit(X, y)  # Fit on full training set
    return model, {'cv_auroc_mean': scores.mean(), 'cv_auroc_std': scores.std(), 'cv_scores': scores}

def train_random_forest(X, y, cv=5):
    """Train Random Forest with stratified k-fold CV.
    Uses class_weight='balanced' for imbalanced data."""
    model = RandomForestClassifier(
        n_estimators=200,
        max_depth=10,
        class_weight='balanced',
        random_state=SEED,
        n_jobs=-1
    )
    skf = StratifiedKFold(n_splits=cv, shuffle=True, random_state=SEED)
    scores = cross_val_score(model, X, y, cv=skf, scoring='roc_auc')
    model.fit(X, y)
    return model, {'cv_auroc_mean': scores.mean(), 'cv_auroc_std': scores.std(), 'cv_scores': scores}

def train_xgboost(X, y, cv=5):
    """Train XGBoost with stratified k-fold CV.
    Uses scale_pos_weight to handle class imbalance."""
    # scale_pos_weight = number of negatives / number of positives
    spw = (y == 0).sum() / max((y == 1).sum(), 1)
    model = xgb.XGBClassifier(
        n_estimators=200,
        max_depth=6,
        learning_rate=0.1,
        scale_pos_weight=spw,
        random_state=SEED,
        eval_metric='logloss',
        use_label_encoder=False
    )
    skf = StratifiedKFold(n_splits=cv, shuffle=True, random_state=SEED)
    scores = cross_val_score(model, X, y, cv=skf, scoring='roc_auc')
    model.fit(X, y)
    return model, {'cv_auroc_mean': scores.mean(), 'cv_auroc_std': scores.std(), 'cv_scores': scores}

In [ ]:
# ============================================================
# Train all three traditional ML models
# ============================================================

print('Training Logistic Regression...')
lr_model, lr_metrics = train_logistic_regression(X_train, y_train)
print(f'  CV AUROC: {lr_metrics["cv_auroc_mean"]:.4f} +/- {lr_metrics["cv_auroc_std"]:.4f}')

print('\nTraining Random Forest...')
rf_model, rf_metrics = train_random_forest(X_train, y_train)
print(f'  CV AUROC: {rf_metrics["cv_auroc_mean"]:.4f} +/- {rf_metrics["cv_auroc_std"]:.4f}')

print('\nTraining XGBoost...')
xgb_model, xgb_metrics = train_xgboost(X_train, y_train)
print(f'  CV AUROC: {xgb_metrics["cv_auroc_mean"]:.4f} +/- {xgb_metrics["cv_auroc_std"]:.4f}')

# Store models for later evaluation
trained_models = {
    'Logistic Regression': lr_model,
    'Random Forest': rf_model,
    'XGBoost': xgb_model,
}

print('\nAll traditional models trained!')

---

<a id="section-6-deep-learning-model"></a>

## 6. Deep Learning Model

### Learning Objectives

By the end of this section, you will:

- Build a feedforward neural network in PyTorch for binary classification
- Implement a training loop with BCE loss, Adam optimizer, and early stopping
- Create PyTorch DataLoaders for batched training
- Visualize training and validation loss curves
- Understand dropout regularization and its role in preventing overfitting

In [ ]:
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

# ============================================================
# Neural Network Architecture
# ============================================================
# A feedforward network with 2 hidden layers.
# Architecture: Input -> 128 -> ReLU -> Dropout -> 64 -> ReLU -> Dropout -> 1 -> Sigmoid
# ============================================================

class MortalityNet(nn.Module):
    """Feedforward neural network for ICU mortality prediction."""

    def __init__(self, n_features, dropout=0.3):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(n_features, 128),  # First hidden layer
            nn.ReLU(),                    # Non-linear activation
            nn.Dropout(dropout),          # Regularization
            nn.Linear(128, 64),           # Second hidden layer
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(64, 1),             # Output layer
            nn.Sigmoid()                  # Probability output [0, 1]
        )

    def forward(self, x):
        return self.network(x).squeeze(-1)

print(f'MortalityNet architecture (input_dim={X_train.shape[1]}):')
print(MortalityNet(X_train.shape[1]))

In [ ]:
def train_neural_network(model, train_loader, val_loader, epochs=100, lr=1e-3, patience=10):
    """Train the neural network with early stopping.

    Uses BCELoss (binary cross-entropy) and Adam optimizer.
    Stops training if validation loss doesn't improve for `patience` epochs.
    Returns the training history (losses per epoch)."""

    criterion = nn.BCELoss()  # Binary cross-entropy for mortality prediction
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    history = {'train_loss': [], 'val_loss': []}
    best_val_loss = float('inf')
    best_state = None
    patience_counter = 0

    for epoch in range(epochs):
        # --- Training phase ---
        model.train()
        train_losses = []
        for X_batch, y_batch in train_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            optimizer.zero_grad()
            y_pred = model(X_batch)
            loss = criterion(y_pred, y_batch)
            loss.backward()
            optimizer.step()
            train_losses.append(loss.item())

        # --- Validation phase ---
        model.eval()
        val_losses = []
        with torch.no_grad():
            for X_batch, y_batch in val_loader:
                X_batch, y_batch = X_batch.to(device), y_batch.to(device)
                y_pred = model(X_batch)
                loss = criterion(y_pred, y_batch)
                val_losses.append(loss.item())

        avg_train = np.mean(train_losses)
        avg_val = np.mean(val_losses)
        history['train_loss'].append(avg_train)
        history['val_loss'].append(avg_val)

        # --- Early stopping check ---
        if avg_val < best_val_loss:
            best_val_loss = avg_val
            best_state = model.state_dict().copy()
            patience_counter = 0
        else:
            patience_counter += 1

        if (epoch + 1) % 10 == 0:
            print(f'  Epoch {epoch+1}/{epochs} - Train Loss: {avg_train:.4f}, Val Loss: {avg_val:.4f}')

        if patience_counter >= patience:
            print(f'  Early stopping at epoch {epoch+1} (no improvement for {patience} epochs)')
            break

    # Restore best model weights
    model.load_state_dict(best_state)
    return history

In [ ]:
# ============================================================
# Create DataLoaders and train the neural network
# ============================================================

# Convert numpy arrays to PyTorch tensors
train_dataset = TensorDataset(
    torch.FloatTensor(X_train),
    torch.FloatTensor(y_train)
)
val_dataset = TensorDataset(
    torch.FloatTensor(X_val),
    torch.FloatTensor(y_val)
)

# DataLoaders handle batching and shuffling
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False)

# Initialize the model and move to device (GPU if available)
nn_model = MortalityNet(n_features=X_train.shape[1]).to(device)

print('Training neural network...')
nn_history = train_neural_network(
    nn_model, train_loader, val_loader,
    epochs=100, lr=1e-3, patience=10
)
print('Neural network training complete!')

In [ ]:
# ============================================================
# Plot training and validation loss curves
# ============================================================

plt.figure(figsize=(8, 5))
plt.plot(nn_history['train_loss'], label='Train Loss', linewidth=2)
plt.plot(nn_history['val_loss'], label='Validation Loss', linewidth=2)
plt.xlabel('Epoch')
plt.ylabel('BCE Loss')
plt.title('Neural Network Training Curves')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f'Best validation loss: {min(nn_history["val_loss"]):.4f}')
print(f'Training stopped at epoch: {len(nn_history["train_loss"])}')

---

<a id="section-7-evaluation--comparison"></a>

## 7. Evaluation & Comparison

### Learning Objectives

By the end of this section, you will:

- Evaluate all models on the held-out test set using AUROC, AUPRC, and F1
- Plot ROC and Precision-Recall curves for visual comparison
- Understand why AUPRC is more informative than AUROC for imbalanced clinical data
- Build a side-by-side comparison table of all model metrics

In [ ]:
from sklearn.metrics import (roc_auc_score, average_precision_score,
    accuracy_score, precision_score, recall_score, f1_score, roc_curve, precision_recall_curve)

def compute_metrics(y_true, y_pred, y_prob):
    """Compute classification metrics for binary mortality prediction."""
    return {
        'auroc': roc_auc_score(y_true, y_prob),
        'auprc': average_precision_score(y_true, y_prob),
        'accuracy': accuracy_score(y_true, y_pred),
        'precision': precision_score(y_true, y_pred, zero_division=0),
        'recall': recall_score(y_true, y_pred, zero_division=0),
        'f1': f1_score(y_true, y_pred, zero_division=0),
    }

def plot_roc_curves(results):
    """Plot ROC curves for all models on one figure."""
    plt.figure(figsize=(8, 6))
    for name, r in results.items():
        fpr, tpr, _ = roc_curve(r['y_true'], r['y_prob'])
        plt.plot(fpr, tpr, label=f"{name} (AUROC={r['metrics']['auroc']:.3f})", linewidth=2)
    plt.plot([0,1],[0,1], 'k--', alpha=0.3)
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title('ROC Curves - All Models')
    plt.legend(loc='lower right')
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

def plot_pr_curves(results):
    """Plot Precision-Recall curves for all models."""
    plt.figure(figsize=(8, 6))
    for name, r in results.items():
        prec, rec, _ = precision_recall_curve(r['y_true'], r['y_prob'])
        plt.plot(rec, prec, label=f"{name} (AUPRC={r['metrics']['auprc']:.3f})", linewidth=2)
    plt.xlabel('Recall')
    plt.ylabel('Precision')
    plt.title('Precision-Recall Curves - All Models')
    plt.legend(loc='upper right')
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

def display_comparison_table(results):
    """Build a side-by-side comparison DataFrame."""
    rows = []
    for name, r in results.items():
        row = {'Model': name}
        row.update({k.upper(): f'{v:.4f}' for k, v in r['metrics'].items()})
        rows.append(row)
    return pd.DataFrame(rows).set_index('Model')

In [ ]:
# ============================================================
# Evaluate all models on the TEST set
# ============================================================

all_results = {}

# Traditional models
for name, model in trained_models.items():
    y_prob = model.predict_proba(X_test)[:, 1]
    y_pred = (y_prob >= 0.5).astype(int)
    metrics = compute_metrics(y_test, y_pred, y_prob)
    all_results[name] = {'y_true': y_test, 'y_prob': y_prob, 'y_pred': y_pred, 'metrics': metrics}
    print(f'{name}: AUROC={metrics["auroc"]:.4f}, AUPRC={metrics["auprc"]:.4f}, F1={metrics["f1"]:.4f}')

# Neural network
nn_model.eval()
with torch.no_grad():
    X_test_tensor = torch.FloatTensor(X_test).to(device)
    nn_prob = nn_model(X_test_tensor).cpu().numpy()
nn_pred = (nn_prob >= 0.5).astype(int)
nn_metrics = compute_metrics(y_test, nn_pred, nn_prob)
all_results['Neural Network'] = {'y_true': y_test, 'y_prob': nn_prob, 'y_pred': nn_pred, 'metrics': nn_metrics}
print(f'Neural Network: AUROC={nn_metrics["auroc"]:.4f}, AUPRC={nn_metrics["auprc"]:.4f}, F1={nn_metrics["f1"]:.4f}')

In [ ]:
# Plot ROC and PR curves
plot_roc_curves(all_results)
plot_pr_curves(all_results)

# Display comparison table
comparison_df = display_comparison_table(all_results)
print('\nModel Comparison (Test Set):')
comparison_df

### Why AUPRC Matters More Than AUROC for Imbalanced Data

In ICU mortality prediction, the positive class (death) is relatively rare (~11-13%). In this setting:

- **AUROC** can be misleadingly high because it accounts for true negatives, which are abundant. A model that predicts "survived" for everyone would still have a decent AUROC.
- **AUPRC** (Area Under the Precision-Recall Curve) focuses on the positive class and is more sensitive to how well the model identifies patients who actually die.

For clinical decision-making, we care most about correctly identifying high-risk patients (recall) without too many false alarms (precision). AUPRC captures this tradeoff directly.

**Clinical interpretation:** Compare the models above. A higher AUPRC means the model is better at distinguishing patients who will die from those who will survive, which is the clinically actionable question.

---

<a id="section-8-interpretability--shap"></a>

## 8. Interpretability / SHAP

### Learning Objectives

By the end of this section, you will:

- Use SHAP (SHapley Additive exPlanations) to explain model predictions
- Generate summary (beeswarm) plots showing global feature importance
- Create force plots explaining individual patient predictions
- Compare SHAP-based importance with built-in tree feature importance
- Assess whether top features are clinically plausible

In [ ]:
import shap

# Initialize SHAP JS visualizations for the notebook
shap.initjs()

def compute_shap_values(model, X, feature_names):
    """Compute SHAP values using TreeExplainer (fast for tree models)."""
    explainer = shap.TreeExplainer(model)
    shap_values = explainer.shap_values(X)
    # For binary classification, some models return a list [class0, class1]
    if isinstance(shap_values, list):
        shap_values = shap_values[1]  # Use positive class
    return shap.Explanation(
        values=shap_values,
        base_values=explainer.expected_value if not isinstance(explainer.expected_value, list) else explainer.expected_value[1],
        data=X,
        feature_names=feature_names
    )

def plot_shap_summary(shap_values):
    """Generate SHAP beeswarm summary plot."""
    shap.summary_plot(shap_values, show=True)

def plot_shap_force(shap_values, idx):
    """Generate SHAP force plot for a single patient."""
    shap.force_plot(shap_values.base_values, shap_values.values[idx],
                    shap_values.data[idx], feature_names=shap_values.feature_names, matplotlib=True)

def get_tree_feature_importance(model, feature_names):
    """Extract built-in feature importance from tree models."""
    importances = model.feature_importances_
    return pd.DataFrame({
        'Feature': feature_names,
        'Importance': importances
    }).sort_values('Importance', ascending=False).reset_index(drop=True)

In [ ]:
# ============================================================
# Compute SHAP values for the best tree-based model
# ============================================================
# We use the XGBoost model (typically best performer) for SHAP.
# TreeExplainer is exact and fast for tree-based models.
# ============================================================

# Use a sample of test data for SHAP (faster computation)
shap_sample_size = min(500, len(X_test))
X_shap = X_test[:shap_sample_size]

print('Computing SHAP values for XGBoost model...')
shap_values = compute_shap_values(xgb_model, X_shap, feature_names)
print('Done!')

# SHAP Summary Plot (beeswarm) - shows global feature importance
print('\nSHAP Summary Plot (global feature importance):')
plot_shap_summary(shap_values)

In [ ]:
# ============================================================
# SHAP Force Plot for a single patient
# ============================================================
# This shows how each feature pushes the prediction higher or
# lower for one specific patient. Red features push toward
# mortality, blue features push toward survival.
# ============================================================

print('SHAP Force Plot for Patient #0 (first test patient):')
plt.figure(figsize=(12, 3))
plot_shap_force(shap_values, idx=0)
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# Compare tree-based feature importance: RF vs XGBoost
# ============================================================

rf_importance = get_tree_feature_importance(rf_model, feature_names)
xgb_importance = get_tree_feature_importance(xgb_model, feature_names)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Random Forest importance (top 15)
top_rf = rf_importance.head(15)
axes[0].barh(top_rf['Feature'], top_rf['Importance'], color='steelblue')
axes[0].set_title('Random Forest - Top 15 Features')
axes[0].set_xlabel('Importance')
axes[0].invert_yaxis()

# XGBoost importance (top 15)
top_xgb = xgb_importance.head(15)
axes[1].barh(top_xgb['Feature'], top_xgb['Importance'], color='darkorange')
axes[1].set_title('XGBoost - Top 15 Features')
axes[1].set_xlabel('Importance')
axes[1].invert_yaxis()

plt.tight_layout()
plt.show()

### Clinical Plausibility of Top Features

The top features identified by SHAP and tree importance should align with clinical knowledge about ICU mortality:

- **Age**: Older patients have higher mortality risk in the ICU
- **Lactate**: Elevated lactate indicates tissue hypoperfusion and is a strong predictor of sepsis-related mortality
- **BUN/Creatinine**: Elevated values indicate renal dysfunction, common in critically ill patients
- **Heart rate / Blood pressure**: Hemodynamic instability (tachycardia, hypotension) signals cardiovascular compromise
- **WBC**: Very high or very low white blood cell counts suggest severe infection or immune dysfunction
- **SpO2**: Low oxygen saturation indicates respiratory failure

If the model's top features don't match clinical expectations, it may indicate data quality issues or overfitting to artifacts rather than true clinical signals. This is why interpretability is essential in healthcare ML.

---

<a id="section-9-bonus-multimodal-extension-optional"></a>

## 9. Bonus: Multimodal Extension (Optional)

### Learning Objectives (Optional Bonus)

By the end of this section, you will:

- Load discharge summary text from MIMIC-III via BigQuery
- Generate fixed-length embeddings using ClinicalBERT (Bio_ClinicalBERT)
- Combine structured clinical features with text embeddings (multimodal fusion)
- Compare multimodal vs structured-only model performance

> **Note:** This section is **optional**. The core tutorial (Sections 1-8) is complete without it.
> ClinicalBERT is used here purely as a **feature extractor** — not as the primary tutorial subject.
> The notebook will continue to work even if this section is skipped or fails.

In [ ]:
# ============================================================
# OPTIONAL: Multimodal Extension
# ============================================================
# This section combines structured clinical data with text
# embeddings from discharge summaries using ClinicalBERT.
# Wrapped in try/except so the notebook works if this fails.
# ============================================================

try:
    from transformers import AutoTokenizer, AutoModel

    def load_discharge_summaries(project_id, dataset='physionet-data.mimiciii_clinical'):
        """Query discharge summaries from BigQuery noteevents table."""
        from google.cloud import bigquery
        client = bigquery.Client(project=project_id)
        query = f"""
            SELECT hadm_id, text
            FROM `{dataset}.noteevents`
            WHERE category = 'Discharge summary'
            AND text IS NOT NULL
        """
        print('Querying discharge summaries from BigQuery...')
        df = client.query(query).to_dataframe()
        # Keep one summary per admission (take the longest)
        df['text_len'] = df['text'].str.len()
        df = df.sort_values('text_len', ascending=False).drop_duplicates('hadm_id').drop('text_len', axis=1)
        print(f'Loaded {len(df)} discharge summaries.')
        return df

    def extract_clinicalbert_embeddings(texts, model_name='emilyalsentzer/Bio_ClinicalBERT', batch_size=16):
        """Generate [CLS] embeddings from ClinicalBERT.
        Returns (n_samples, 768) numpy array."""
        tokenizer = AutoTokenizer.from_pretrained(model_name)
        model = AutoModel.from_pretrained(model_name).to(device)
        model.eval()

        all_embeddings = []
        for i in range(0, len(texts), batch_size):
            batch = texts[i:i+batch_size]
            # Truncate to 512 tokens (BERT max length)
            encoded = tokenizer(batch, padding=True, truncation=True,
                                max_length=512, return_tensors='pt').to(device)
            with torch.no_grad():
                outputs = model(**encoded)
            # Use [CLS] token embedding (first token)
            cls_emb = outputs.last_hidden_state[:, 0, :].cpu().numpy()
            all_embeddings.append(cls_emb)
            if (i // batch_size) % 10 == 0:
                print(f'  Processed {min(i+batch_size, len(texts))}/{len(texts)} texts')

        return np.vstack(all_embeddings)

    def build_multimodal_features(X_structured, X_text):
        """Concatenate structured features with text embeddings."""
        return np.hstack([X_structured, X_text])

    print('Multimodal functions defined successfully.')

except ImportError as e:
    print(f'Transformers library not available: {e}')
    print('Skipping multimodal section.')

In [ ]:
# ============================================================
# Load discharge summaries and generate embeddings
# ============================================================

try:
    # Load discharge summaries
    discharge_df = load_discharge_summaries(PROJECT_ID)

    # Match summaries to our ICU stays via hadm_id
    icu_with_notes = icu_encoded.merge(discharge_df, on='hadm_id', how='inner')
    print(f'ICU stays with discharge summaries: {len(icu_with_notes)}')

    # Generate ClinicalBERT embeddings (this may take a few minutes)
    print('\nGenerating ClinicalBERT embeddings...')
    texts = icu_with_notes['text'].tolist()
    text_embeddings = extract_clinicalbert_embeddings(texts)
    print(f'Embedding shape: {text_embeddings.shape}')

    # Build multimodal features
    X_multi, y_multi = build_feature_matrix(icu_with_notes, feature_cols)
    X_combined = build_multimodal_features(X_multi, text_embeddings)
    print(f'Combined feature matrix: {X_combined.shape}')

    # Split and train a model on combined features
    X_train_m, X_test_m, y_train_m, y_test_m = train_test_split(
        X_combined, y_multi, test_size=0.2, random_state=SEED, stratify=y_multi
    )

    # Train Logistic Regression on multimodal features
    print('\nTraining Logistic Regression on multimodal features...')
    lr_multi = LogisticRegression(max_iter=1000, class_weight='balanced', random_state=SEED)
    lr_multi.fit(X_train_m, y_train_m)
    y_prob_multi = lr_multi.predict_proba(X_test_m)[:, 1]
    y_pred_multi = (y_prob_multi >= 0.5).astype(int)
    multi_metrics = compute_metrics(y_test_m, y_pred_multi, y_prob_multi)

    # Compare with structured-only LR
    lr_struct = LogisticRegression(max_iter=1000, class_weight='balanced', random_state=SEED)
    lr_struct.fit(X_train_m[:, :len(feature_cols)], y_train_m)
    y_prob_struct = lr_struct.predict_proba(X_test_m[:, :len(feature_cols)])[:, 1]
    y_pred_struct = (y_prob_struct >= 0.5).astype(int)
    struct_metrics = compute_metrics(y_test_m, y_pred_struct, y_prob_struct)

    print(f'\nStructured-only LR:  AUROC={struct_metrics["auroc"]:.4f}, AUPRC={struct_metrics["auprc"]:.4f}')
    print(f'Multimodal LR:       AUROC={multi_metrics["auroc"]:.4f}, AUPRC={multi_metrics["auprc"]:.4f}')
    improvement = multi_metrics['auroc'] - struct_metrics['auroc']
    print(f'AUROC improvement:   {improvement:+.4f}')

except Exception as e:
    print(f'Multimodal section skipped or failed: {e}')
    print('This is OK -- the core tutorial is complete without this section.')

### Multimodal Approach Explained

The multimodal extension combines two types of clinical data:

1. **Structured data** (vitals, labs, demographics) — numeric features we engineered in Sections 3-4
2. **Unstructured text** (discharge summaries) — free-text clinical narratives encoded as 768-dimensional vectors by ClinicalBERT

By concatenating these feature types, the model can leverage both quantitative measurements and qualitative clinical observations. Discharge summaries often contain information not captured in structured data, such as:
- Clinical reasoning and differential diagnoses
- Social factors affecting prognosis
- Nuanced descriptions of disease trajectory

> **Note:** ClinicalBERT is used here as a **feature extractor** (generating embeddings), not as the primary subject of this tutorial. For a deep dive into ClinicalBERT itself, see the course's ClinicalBERT notebooks.

---

<a id="section-10-conclusion--references"></a>

## 10. Conclusion & References

### Key Takeaways

In this tutorial, we built a complete machine learning pipeline for ICU mortality prediction using MIMIC-III data:

1. **Data Pipeline**: Loaded structured clinical data from Google BigQuery, merged five MIMIC-III tables, and filtered to the first 24 hours of each ICU stay
2. **Preprocessing**: Handled missing data (median imputation), clipped physiologically implausible outliers, and normalized features
3. **Feature Engineering**: Encoded categorical variables and built a feature matrix with ~61 structured features
4. **Traditional ML**: Trained Logistic Regression, Random Forest, and XGBoost with stratified cross-validation and class weighting
5. **Deep Learning**: Built a feedforward neural network in PyTorch with early stopping
6. **Evaluation**: Compared all models using AUROC, AUPRC, and F1 — with emphasis on AUPRC for imbalanced clinical data
7. **Interpretability**: Used SHAP to explain predictions and verify clinical plausibility of top features
8. **Multimodal (Bonus)**: Combined structured features with ClinicalBERT discharge summary embeddings

The tree-based models (Random Forest, XGBoost) typically perform well on structured clinical data, while the neural network may benefit more from the multimodal extension where it can learn complex interactions between structured and text features.

### Limitations & Future Work

**Limitations:**
- Single-center data (MIMIC-III is from Beth Israel Deaconess Medical Center) — results may not generalize to other hospitals
- Only first 24 hours of data used — earlier or later prediction windows may yield different results
- Simple temporal aggregation (mean) — more sophisticated time-series features (trends, variability) could improve performance
- No missing data mechanism analysis — we assumed data is missing at random, which may not hold clinically
- Limited hyperparameter tuning — production models would benefit from systematic optimization

**Future directions:**
- Use recurrent neural networks (LSTM/GRU) to model temporal patterns in vital signs
- Apply the pipeline to multi-center data (eICU) for external validation
- Explore fairness across demographic subgroups (age, gender, insurance type)
- Implement calibration analysis to ensure predicted probabilities are clinically meaningful
- Add attention mechanisms to the multimodal model for interpretable text-structure fusion

### References

1. Johnson, A.E.W., Pollard, T.J., Shen, L. et al. **MIMIC-III, a freely accessible critical care database.** *Scientific Data* 3, 160035 (2016). https://doi.org/10.1038/sdata.2016.35

2. Alsentzer, E., Murphy, J.R., Boag, W. et al. **Publicly Available Clinical BERT Embeddings.** *Proceedings of the 2nd Clinical Natural Language Processing Workshop* (2019). https://arxiv.org/abs/1904.03323

3. Lundberg, S.M. & Lee, S.I. **A Unified Approach to Interpreting Model Predictions.** *NeurIPS* (2017). https://arxiv.org/abs/1705.07874

4. Chen, T. & Guestrin, C. **XGBoost: A Scalable Tree Boosting System.** *KDD* (2016). https://arxiv.org/abs/1603.02754

5. Harutyunyan, H., Khachatrian, H., Kale, D.C. et al. **Multitask learning and benchmarking with clinical time series data.** *Scientific Data* 6, 96 (2019). https://doi.org/10.1038/s41597-019-0103-9

6. Purushotham, S., Meng, C., Che, Z. & Liu, Y. **Benchmarking deep learning models on large healthcare datasets.** *Journal of Biomedical Informatics* 83, 112-134 (2018).

---

*Tutorial created for Healthcare ML course. Data accessed via PhysioNet under the MIMIC-III Data Use Agreement.*